In [1]:
import pandas as pd

In [26]:
a = pd.read_csv("data_ws2026/BAC.csv")
a["Ticker"] = "BAC"
b = pd.read_csv("data_ws2026/LRCX.csv")
b["Ticker"] = "LRCX"
c = pd.read_csv("data_ws2026/NTAP.csv")
c["Ticker"] = "NTAP"
d = pd.read_csv("data_ws2026/RTX.csv")
d["Ticker"] = "RTX"

In [27]:
df = pd.concat([a,b,c,d])
df.Date = pd.to_datetime(df.Date)
df = df.set_index("Date", append=False).sort_index()
df

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker
Date,,,,,,,,
2022-01-03,40.087,41.331,39.989,41.056,58587900,0.0,0.0,BAC
2022-01-03,68.701,69.647,67.815,69.114,10602000,0.0,0.0,LRCX
2022-01-03,77.769,78.726,77.760,78.573,5169200,0.0,0.0,RTX
2022-01-03,82.431,83.121,82.028,82.727,1012800,0.0,0.0,NTAP
2022-01-04,79.386,81.373,79.232,81.328,6044200,0.0,0.0,RTX
...,...,...,...,...,...,...,...,...
2025-12-29,55.224,55.362,54.398,54.476,21040100,0.0,0.0,BAC
2025-12-30,176.018,176.835,173.076,173.266,5670800,0.0,0.0,LRCX
2025-12-30,54.614,54.762,54.299,54.408,17427600,0.0,0.0,BAC


In [30]:
# 1.
df["diff"] = df.High - df.Low

# no company with zero difference for all days, ie, >1 unique values for all
df.groupby("Ticker")["diff"].nunique()

Ticker
BAC     729
LRCX    915
NTAP    852
RTX     860
Name: diff, dtype: int64

In [33]:
# none whatsoever
df[df["diff"] == 0.0]
# meaning = the stock price stayed the same for the whole day, the stock exchange was perhaps closed

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Ticker,diff
Date,,,,,,,,,


In [35]:
# 2.
g = df.groupby("Ticker")
pd.DataFrame({
    "highest_high": g["High"].max(),
    
    "lowest_low": g["Low"].min(),
  
})

,highest_high,lowest_low
Ticker,,
BAC,55.657,23.229
LRCX,179.268,28.841
NTAP,130.747,53.433
RTX,185.983,64.579


In [53]:
# 4.
g = df.groupby("Ticker")
pd.DataFrame({
    "highest_high": g["High"].idxmax(),
    "lowest_low": g["Low"].idxmin(),
})

t = ["BAC", "LRCX", "NTAP", "RTX"]
mx = []
mn = []
for s in t:
    mx = df[(df.Ticker == s)].sort_values("High", ascending=False).iloc[1:2].index
    print(f"second max for {s}:{mx}")
    mn = df[(df.Ticker == s)].sort_values("Low", ascending=True).iloc[1:2].index
    print(f"second min for {s}:{mn}")
    print("---")



second max for BAC:DatetimeIndex(['2025-12-24'], dtype='datetime64[us]', name='Date', freq=None)
second min for BAC:DatetimeIndex(['2023-10-25'], dtype='datetime64[us]', name='Date', freq=None)
---
second max for LRCX:DatetimeIndex(['2025-12-29'], dtype='datetime64[us]', name='Date', freq=None)
second min for LRCX:DatetimeIndex(['2022-10-17'], dtype='datetime64[us]', name='Date', freq=None)
---
second max for NTAP:DatetimeIndex(['2024-07-10'], dtype='datetime64[us]', name='Date', freq=None)
second min for NTAP:DatetimeIndex(['2022-12-20'], dtype='datetime64[us]', name='Date', freq=None)
---
second max for RTX:DatetimeIndex(['2025-12-24'], dtype='datetime64[us]', name='Date', freq=None)
second min for RTX:DatetimeIndex(['2023-10-04'], dtype='datetime64[us]', name='Date', freq=None)
---


In [56]:
# 3.
import numpy as np
df["close_log"] = df.Close.apply(np.log)
df["ret"] = df.groupby("Ticker")["close_log"].pct_change()
df.groupby("Ticker")["ret"].agg(["mean", "std", "min", "max", "skew"])

,mean,std,min,max,skew
Ticker,,,,,
BAC,0.000085,0.004916,-0.031676,0.021929,-0.072887
LRCX,0.000221,0.007024,-0.028734,0.040525,-0.041100
NTAP,0.000067,0.004595,-0.035700,0.037582,0.364589
RTX,0.000181,0.003301,-0.023904,0.017171,-0.636816


In [62]:
# 5.
weekly = df.groupby("Ticker")["Volume"].resample("W").sum()
weekly.groupby("Ticker").median()

Ticker
BAC     193509800.0
LRCX     56167900.0
NTAP      8371300.0
RTX      24591500.0
Name: Volume, dtype: float64

In [82]:
# 6.
print(df[df.Ticker == "BAC"].close_log.cumsum().sum())

print(df[df.Ticker == "LRCX"].close_log.cumsum().sum())

print(df[df.Ticker == "NTAP"].close_log.cumsum().sum())

print(df[df.Ticker == "RTX"].close_log.cumsum().sum())

# == BAC is lowest

1749706.542542221
2033453.4734450607
2168353.3762275996
2262971.9528351375


In [89]:
# PART 2
df2 = pd.read_csv("data_ws2026/companies_no_subindustry.csv")
df2[df2["included"].isna()].shape

,Symbol,Security,GICS_Sector,hq,included,Founded
28,AMD,AMD,Information Technology,"Santa Clara, California",NaN,1969
125,ED,Consolidated Edison,Utilities,"New York City, New York",NaN,1823
130,GLW,Corning Inc.,Information Technology,"Corning, New York",NaN,1851
140,DHR,Danaher Corporation,Health Care,"Washington, D.C.",NaN,1969
141,DRI,Darden Restaurants,Consumer Discretionary,"Orlando, Florida",NaN,1938
155,D,Dominion Energy,Utilities,"Richmond, Virginia",NaN,1983
164,ETN,Eaton Corporation,Industrials,"Dublin, Ireland",NaN,1911
197,FITB,Fifth Third Bank,Financials,"Cincinnati, Ohio",NaN,1858
199,FE,FirstEnergy,Utilities,"Akron, Ohio",NaN,1997
210,BEN,Franklin Templeton,Financials,"San Mateo, California",NaN,1947
